In [1]:
from typing import List
from pydantic import BaseModel, Field

from langchain.chat_models import init_chat_model
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate

"""
표준 정답지(Golden Set)
- 질문 (user_input)
- 모범정답 (reference)
- 근거문서 (retrieved_contexts)
"""

class RagEvalSample(BaseModel):
    user_input: str = Field(description="제공된 근거 문서들을 바탕으로 사용자가 질의할 법한 질문")
    reference: str = Field(description="근거 문서에 기반한 모범정답(Ground Truth)")
    retrieved_contexts: List[str] = Field(
        description="해당 질문과 정답을 도출하는 데 실제로 사용된 1개 이상의 원본 문서 내용"
    )

class RagEvalDataset(BaseModel):
    samples: List[RagEvalSample] = Field(
        description="근거 문서로부터 생성된 평가 샘플 목록"
    )

source_docs: List[Document] = [
    Document(
        page_content="원격 근무 지원 규정: 주 2회 재택근무가 가능하며, 원격 근무 당일은 오전 9시까지 사내 메신저 근태 채널에 출근 등록을 완료해야 합니다.",
        metadata={"source": "근태규정.pdf", "chunk_id": 1}
    ),
    Document(
        page_content="복리후생 지원 규정: 재택근무 지원금은 월 50,000원이 지급되며, 매월 25일 급여일에 합산되어 지급됩니다.",
        metadata={"source": "복리후생.pdf", "chunk_id": 2}
    ),
    Document(
        page_content="수습 사원 근무 가이드: 신규 입사자는 입사 후 3개월간 수습 기간을 거치며, 수습 기간 중에는 원격 근무 및 관련 지원금 신청이 제한됩니다.",
        metadata={"source": "인사핸드북.pdf", "chunk_id": 3}
    ),
]

def format_documents(docs):
    _docs = [f"[문서 {doc.metadata.get("chunk_id")}]\n{doc.page_content.strip()}" for doc in source_docs]

    return "\n\n".join(_docs)